This notebook is used to lemmatize words in DoReCo and data of other predictors.

In [11]:
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from pathlib import Path
import os

Print version numbers for reproducibility

In [12]:
%load_ext watermark
%watermark
%watermark --iversions

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark
Last updated: 2025-09-25T10:06:10.629306+10:00

Python implementation: CPython
Python version       : 3.11.5
IPython version      : 8.12.3

Compiler    : MSC v.1936 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 154 Stepping 4, GenuineIntel
CPU cores   : 12
Architecture: 64bit

pandas: 2.2.3
nltk  : 3.8.1



In [13]:
project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

file_path = os.path.join(project_folder, "data", "doreco_counts.csv")

focus_list = set()
with open(file_path, 'r', encoding='utf-8') as f:
    header = f.readline().strip().split(",")
    word_index = header.index("word")

    for line in f:
        parts = line.strip().split(",")
        if len(parts) > word_index:
            focus_list.add(parts[word_index])

In [14]:
total_words = len(focus_list)
present_in_wordnet = 0
absent_from_wordnet = 0

for word in focus_list:
    if wn.synsets(word):
        present_in_wordnet += 1
    else:
        absent_from_wordnet += 1

print("Total words/numbers in DoReCo:", total_words)
print("Words/numbers present in WordNet:", present_in_wordnet)
print("Words/numbers absent from WordNet:", absent_from_wordnet)

Total words/numbers in DoReCo: 27250
Words/numbers present in WordNet: 12767
Words/numbers absent from WordNet: 14483


In [15]:
# lemmatize each word in the focus list
lemmatizer = WordNetLemmatizer()
# Filter and lemmatize words that appear in WordNet
lemmatized_words = [
    (word, lemmatizer.lemmatize(word))
    for word in focus_list
    if wn.synsets(word)  # Only include words present in WordNet
]

df = pd.DataFrame(lemmatized_words, columns=['original_word', 'lemmatized_word'])
df = df.sort_values(by="original_word").reset_index(drop=True)

In [16]:
df.tail(10)

,original_word,lemmatized_word
12757,yucca,yucca
12758,yurt,yurt
12759,yurts,yurt
12760,zeal,zeal
12761,zebras,zebra
12762,zigzag,zigzag
12763,zombies,zombie
12764,zona,zona
12765,zoo,zoo
12766,zu,zu


In [17]:
lemma_doreco_file = os.path.join(project_folder, "data", "lemma_doreco.tsv")
df.to_csv(lemma_doreco_file, sep='\t', index=False, encoding='utf-8')

Lemmatize words in other data sets.

In [18]:
project_folder = Path.cwd()
if project_folder.name == "preprocessing":
    project_folder = project_folder.parent

swow_path = project_folder / "rawdata" / "swow" / "responseStats.SWOW-EN.20180827.csv"
cd_path = project_folder / "data" / "english_counts.csv"
output_path = project_folder / "data" / "lemma_others.tsv"

# Load the datasets
swow = pd.read_csv(swow_path, encoding='ISO-8859-1')
cd = pd.read_csv(cd_path, encoding='utf-8')

# Combine the words into a unique set
focus_list = set(swow['response']).union(cd['term'])
focus_list = {str(word) for word in focus_list if pd.notna(word)}

total_words = len(focus_list)
present_in_wordnet = 0
absent_from_wordnet = 0

for word in focus_list:
    if wn.synsets(word):
        present_in_wordnet += 1
    else:
        absent_from_wordnet += 1

print("Total words/numbers:", total_words)
print("Words/numbers present in WordNet:", present_in_wordnet)
print("Words/numbers absent from WordNet:", absent_from_wordnet)

Total words/numbers: 885365
Words/numbers present in WordNet: 96402
Words/numbers absent from WordNet: 788963


In [19]:
# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Filter and lemmatize words that appear in WordNet
lemmatized_words = [
    (word, lemmatizer.lemmatize(word))
    for word in focus_list
    if wn.synsets(word)  # Only include words present in WordNet
]

# Create a DataFrame and save as a TSV file
df = pd.DataFrame(lemmatized_words, columns=['original_word', 'lemmatized_word'])
df = df.sort_values(by="original_word").reset_index(drop=True)
df.to_csv(output_path, sep='\t', index=False, encoding='utf-8')

In [20]:
df.tail(10)

,original_word,lemmatized_word
96392,zus,zu
96393,zweig,zweig
96394,zwingli,zwingli
96395,zydeco,zydeco
96396,zygodactyl,zygodactyl
96397,zygomatic,zygomatic
96398,zygote,zygote
96399,zygotes,zygote
96400,zygotic,zygotic
96401,zymurgy,zymurgy
